# 1. Design a banking system with: 


In [26]:
# • An abstract base class Account with deposit(), withdraw(), calculate_interest().
# • Subclasses: SavingsAccount, CurrentAccount, FixedDepositAccount. 
# • Each account must:  Encapsulate balance (private) 
#   Provide controlled access through properties 
#   Override interest calculation differently 
# • Include a static method to validate amount. 
# • Include a class method to update bank-wide interest policies. 
# Demonstrate: 
# • Polymorphic behavior by iterating through all account types
# • Preventing direct access to balance 
# • Multiple interest strategies 
from abc import ABC,abstractmethod
class Account(ABC):
    interest_rates = {
            "savings": 5,
            "current": 2,
            "fixed": 10
        }
    def __init__(self,balance):
        self.__balance=balance
    @property
    def balance(self):
        return self.__balance
    @balance.setter
    def balance(self,new_balance):
        if new_balance>=0:
            self.__balance=new_balance
        else:
            raise ValueError("Balance cannot be negative")
    @staticmethod
    def check_amount(amount):
        return amount>0
    @classmethod
    def change_interest(cls,acc_type,new_rate):
        if acc_type not in Account.interest_rates:
            raise ValueError("Invalid Acc Type")
        elif new_rate<0:
            raise ValueError("Interest Rate Cant be Negatiive")
        cls.interest_rates[acc_type]=new_rate
        print(f"{acc_type} Interest Rate Updated Successfully to {new_rate}")

    def deposit(self,amount,acc_type):
        if self.check_amount(amount):
            self.balance = self.balance + amount
            print(f"Dear User! Your {acc_type} Account Has Been Credited With RS.",amount)
        else:
            raise ValueError("Enter A Valid Amount To Deposit")
        
    def withdraw(self,amount,acc_type):
        if self.check_amount(amount) and amount<=self.__balance:
            self.balance = self.balance - amount
            print(f"Dear User! Your {acc_type} Account Has Been Debited With RS.",amount)
        elif amount>self.__balance:
            raise ValueError("Insufficients Funds")
        else:
            raise ValueError("Invalid Amount")
        
    @abstractmethod
    def calculate_interest(self):
            pass
    
class SavingsAccount(Account):
    def __init__(self, balance):
        super().__init__(balance)
    def calculate_interest(self):
        rate=Account.interest_rates['savings']
        return self.balance*rate/100
   
class CurrentAccount(Account):
    def __init__(self, balance):
       super().__init__(balance)
    def calculate_interest(self):
        if self.balance>=50000:
            rate=Account.interest_rates['current']
            return self.balance*rate/100
        return 0
    
class FixedDepositAccount(Account):
    def __init__(self,balance,years):
        super().__init__(balance)
        self.years=years
    def calculate_interest(self):
        rate=Account.interest_rates['fixed']/100
        return self.balance*((1+rate)**self.years-1)

#polymorphism
accounts = [
    SavingsAccount(100000),
    CurrentAccount(60000),
    FixedDepositAccount(50000, 2)
]
for account in accounts:
    print(
        type(account).__name__,
        "| Balance:", account.balance,
        "| Interest:", round(account.calculate_interest(), 2)
    )
    
print("\n--- Deposit and Withdrawal ---")
accounts[1].deposit(5000,'Current')
accounts[1].withdraw(2000,'Current')
print("Updated balance:", accounts[0].balance)

print("\n--- Deposit and Withdrawal ---")
accounts[2].deposit(5000,'FixedDeposit')
accounts[2].withdraw(2000,'FixedDeposit')
print("Updated balance:", accounts[2].balance)

print("\n--- Update Bank Interest Policy ---")
Account.change_interest("current", 7)
print("New current interest:", accounts[1].calculate_interest())

print("\n--- Private Balance Demonstration ---")
print("Access through property:", accounts[0].balance)
print("Direct __balance access:", hasattr(accounts[0], "__balance"))

SavingsAccount | Balance: 100000 | Interest: 5000.0
CurrentAccount | Balance: 60000 | Interest: 1200.0
FixedDepositAccount | Balance: 50000 | Interest: 10500.0

--- Deposit and Withdrawal ---
Dear User! Your Current Account Has Been Credited With RS. 5000
Dear User! Your Current Account Has Been Debited With RS. 2000
Updated balance: 100000

--- Deposit and Withdrawal ---
Dear User! Your FixedDeposit Account Has Been Credited With RS. 5000
Dear User! Your FixedDeposit Account Has Been Debited With RS. 2000
Updated balance: 53000

--- Update Bank Interest Policy ---
current Interest Rate Updated Successfully to 7
New current interest: 4410.0

--- Private Balance Demonstration ---
Access through property: 100000
Direct __balance access: False


In [ ]:
# 2. Build: 
# • Vehicle base class 
# • Car, Bike, Auto subclasses 
# • A Driver class that contains a Vehicle 
# • A Ride class that: 
#  Calculates fare differently depending on the type of vehicle (polymorphism) 
#  Stores driver + vehicle combination 
#  Protects internal fare formula through encapsulation 
#  Also: Use __str__ to print readable ride summaries. 
#  Show how composition + polymorphism interact. 
class Vehicle:
    def calc_fare(self):
        pass
class Car(Vehicle):
    def calc_fare(self):
        return 250
class Bike(Vehicle):
    def calc_fare(self):
        return 120
class Auto(Vehicle):
    def calc_fare(self):
        return 80
class Driver:
    def __init__(self,name,vehicle):
        self.name=name
        self.vehicle=vehicle
class Ride:
    def __init__(self, driver):
        self.driver = driver
        self.__fare = self.__calc_fare()
    def __calc_fare(self):
        return self.driver.vehicle.calc_fare()
    def __str__(self):
        return f"Driver: {self.driver.name}, Vehicle: {type(self.driver.vehicle).__name__}, Fare: ₹{self.__fare}"
car = Car()
bike = Bike()
auto = Auto()
d1 = Driver("Naresh", car)
d2 = Driver("Akhil", bike)
d3 = Driver("Madhu", auto)
r1 = Ride(d1)
r2 = Ride(d2)
r3 = Ride(d3)
print(r1)
print(r2)
print(r3)

Driver: Naresh, Vehicle: Car, Fare: ₹250
Driver: Akhil, Vehicle: Bike, Fare: ₹120
Driver: Madhu, Vehicle: Auto, Fare: ₹80


In [14]:
# 3. Create: 
# • Abstract class PaymentMethod with pay(), validate() 
# • Subclasses: CardPayment, WalletPayment, UPIPayment 
# • Encapsulate user balance
# • Use @property to control reading available funds 
# • Overload + operator to combine two payment methods into “split payment” 
# • Demonstrate polymorphism through a checkout loop. 
from abc import ABC,abstractmethod
class PaymentMethod(ABC):
    def __init__(self,balance):
        self.__balance=balance
    @property
    def balance(self):
        return self.__balance
    def _deduct_balance(self, amount):
        self.__balance -= amount
    @abstractmethod
    def pay(self):
        pass
    @abstractmethod
    def validate(self):
        pass
    def __add__(self,other):
        return SplitPayment(self,other)
class SplitPayment:
    def __init__(self, payment1, payment2):
        self.payment1 = payment1
        self.payment2 = payment2
    def pay(self, amount):
        half = amount / 2
        self.payment1.pay(half)
        self.payment2.pay(half)
        print(f"Split payment of ₹{amount} completed")
class CardPayment(PaymentMethod):
    card_number=350176440806
    def pay(self,amount):
        if amount<=self.balance:
            self._deduct_balance(amount)
            print(f"Card payment of ₹{amount} successful")
            print("Your Current Balance is:",self.balance)
        else:
            raise ValueError("Insufficient Amount")
    def validate(self,no):
        if no==CardPayment.card_number:
            print("Card is Valid")
        else:
            print("Invalid Card")
class WalletPayment(PaymentMethod):
    wall_number=433444
    def pay(self,amount):
        if amount<=self.balance:
            self._deduct_balance(amount)
            print(f"Wallet payment of ₹{amount} successful")
            print("Your Current Balance is:",self.balance)
        else:
            raise ValueError("Insufficient Amount")
    def validate(self,no):
        if no==WalletPayment.wall_number:
            print("Wallet is Valid")
        else:
            print("Wallet is not Valid")
class UPIPayment(PaymentMethod):
    upi_id=4444
    def pay(self,amount):
        if amount<=self.balance:
            self._deduct_balance(amount)
            print(f"UPI payment of ₹{amount} successful")
            print("Your Current Balance is:",self.balance)
        else:
            raise ValueError("Insufficient Amount")
    def validate(self,no):
        if no==UPIPayment.upi_id:
            print("UPI is Valid")
        else:
            print("UPI is not Valid")
cp=CardPayment(100000)
wp=WalletPayment(200000)
up=UPIPayment(50000)
payments=[cp,wp,up]
for payment in payments:
    payment.pay(1000)
# print("split-payment(operator overloading):",cp+wp)
print()
split = cp + wp
split.pay(2000)


Card payment of ₹1000 successful
Your Current Balance is: 99000
Wallet payment of ₹1000 successful
Your Current Balance is: 199000
UPI payment of ₹1000 successful
Your Current Balance is: 49000

Card payment of ₹1000.0 successful
Your Current Balance is: 98000.0
Wallet payment of ₹1000.0 successful
Your Current Balance is: 198000.0
Split payment of ₹2000 completed


                         multilevel inheritance + abstraction + encapsulation + method overriding + super()                           

In [21]:
# 4. Create classes: 
# • Person → base • MedicalStaff(Person) • Doctor(MedicalStaff) • Surgeon(Doctor) 
# Requirements: • Hide sensitive data (e.g., salary, patient notes)
# • Abstract method perform_duty()
# • Each level overrides the method with more specific behavior 
# • Use super() to chain constructor calls 
from abc import ABC,abstractmethod
class Person(ABC):
    def __init__(self,name,age):
        self.name=name
        self.age=age
    @abstractmethod
    def perform_duty(self):
        pass
class MedicalStaff(Person):
    def __init__(self, name, age,department,salary):
        super().__init__(name, age)
        self.department=department
        self.__salary=salary
    def perform_duty(self):
        print("Medical staff performs healthcare-related duties")
class Doctor(MedicalStaff):
    def __init__(self, name, age, department, salary,specialization,patient_notes):
        super().__init__(name, age, department, salary)
        self.specialization=specialization
        self.__patient_notes=patient_notes
    def perform_duty(self):
        print("Doctor diagnoses and treats patients.")
class Surgeon(Doctor):
    def __init__(self, name, age, department, salary, specialization, patient_notes,surgery_specialization):
        super().__init__(name, age, department, salary, specialization, patient_notes)
        self.surgery_specialization=surgery_specialization
    def perform_duty(self):
        print("Surgeon performs surgical procedures.")

m=MedicalStaff("Ravi", 35, "Emergency", 50000)
d=Doctor("Arun", 40, "Cardiology", 100000, "Cardiologist", "Patient notes")
s=Surgeon("Kiran", 45, "Surgery", 150000, "General Surgery", "Patient notes", "Heart Surgery")
m.perform_duty()
d.perform_duty()
s.perform_duty()
# print(s.__dict__)
# print(s.department)

Medical staff performs healthcare-related duties
Doctor diagnoses and treats patients.
Surgeon performs surgical procedures.


                                Hybrid inheritance + MRO + method overriding + encapsulation.                                        

In [ ]:
# 5. Classes:
# • User 
# • Instructor(User) 
# • Student(User) 
# • TeachingAssistant(Student, Instructor)
# Requirements: 
# • Track course assignments privately 
# • Ensure TAs override submit_work() and grade_work()
# • Print MRO and explain how Python resolves conflicts 
class User:
    def __init__(self,name,email):
        self.name=name
        self.email=email
class Instructor(User):
    def grade_work(self):
        print("Instructor Grades Your Assignment Based on your work")
class Student(User):
    def submit_work(self):
        print("Students Submits Assignments")
class TeachingAssistant(Student,Instructor):
    def __init__(self, name, email,course_assignments):
        super().__init__(name, email)
        self.__course_assignments=course_assignments
    # __course_assignments = ["Python", "SQL", "OOP"]
    @property
    def course(self):
        return self.__course_assignments
    @course.setter
    def course(self,new):
        self.__course_assignments.extend(new)
    def submit_work(self):
        print("Teaching Assistant Submits Assignments")
    def grade_work(self):
        print("Teaching Assistant Grades Your Assignments")
ta=TeachingAssistant('naresh','nareshnetha090@gmail.com',["Python", "SQL", "OOP"])
ta.submit_work()
ta.grade_work()
print(TeachingAssistant.mro())
s=Student('madhu','maddy@123@gmail.com')
s.submit_work()
ta.course=['ml','devoops','cloud computing']
ta.course

Teaching Assistant Submits Assignments
Teaching Assistant Grades Your Assignments
[<class '__main__.TeachingAssistant'>, <class '__main__.Student'>, <class '__main__.Instructor'>, <class '__main__.User'>, <class 'object'>]
Students Submits Assignments


['Python', 'SQL', 'OOP', 'ml', 'devoops', 'cloud computing']

                             composition/containment + operator overloading + class methods + encapsulation                                

In [54]:
# 6. Create:
# • Product class with private price and quantity
# • Warehouse class containing multiple products 
# • Overload:  + to merge warehouses , len() to return number of unique products , in operator to check if product exists
# • Provide class method to track total warehouses created 
class Product:
    def __init__(self,name,price,quantity):
        self.name=name
        self.__price=price
        self.__quantity=quantity
class Warehouse:
    total_ware_houses=0
    def __init__(self):
        self.products=[]
        Warehouse.total_ware_houses+=1
    @classmethod
    def total_warehouse(cls):
        return cls.total_ware_houses
    def add_products(self,new):
        self.products.append(new)
    def __add__(self, other):
        new_warehouse = Warehouse()
        new_warehouse.products=self.products+other.products
        return new_warehouse
    def __len__(self):
        unique_products=set()
        for product in self.products:
            unique_products.add(product.name)
        return len(unique_products)
    def __contains__(self, item):
        if item in self.products:
            return True
        else:
            return False
        
p1 = Product("TV", 30000, 5)
p2 = Product("AC", 40000, 3)
p3 = Product("Washing Machine", 25000, 4)
p4 = Product("Cooler", 10000, 6)
p5 = Product("TV", 30000, 2)

w1 = Warehouse()
w2 = Warehouse()

w1.add_products(p1)
w1.add_products(p1)
w1.add_products(p2)
w1.add_products(p3)
w2.add_products(p4)
w2.add_products(p5)

print("Unique products in w1:", len(w1))
print("Unique products in w2:", len(w2))

print("p1 exists in w1:", p1 in w1)
print("p4 exists in w1:", p4 in w1)

w3 = w1 + w2
print("Unique products in merged warehouse:", len(w3))

print("Total warehouses:", Warehouse.total_warehouse())
w1=Warehouse()

Unique products in w1: 3
Unique products in w2: 2
p1 exists in w1: True
p4 exists in w1: False
Unique products in merged warehouse: 4
Total warehouses: 3


                  Abstract class → inheritance → overriding → encapsulation → true polymorphism → duck typing.                            

In [ ]:
# 7. Design:
# • Abstract class MediaFile with play(), stop() 
# • Subclasses: MP3File, MP4File, WAVFile 
# • Private file path validation done internally 
# • A function start_player(media) that works with ANY object that has play() (duck typing)
# Demonstrate mixing true polymorphism + duck typing. 
from abc import ABC,abstractmethod
class MediaFile(ABC):
    def __init__(self, file_path):
        self.__file_path = file_path
        self._validate_path()
    def _validate_path(self):
        if not self.__file_path:
            raise ValueError("File path cannot be empty")
    @abstractmethod
    def play(self):
        pass
    @abstractmethod
    def stop(self):
        pass
class MP3File(MediaFile):
    def play(self):
        print("Playing is mp3")
    def stop(self):
        print("Stops Playing in mp3")
   
class MP4File(MediaFile):
    def play(self):
        print("Playing in mp4")
    def stop(self):
        print("Stops Playing in mp4")
        
class WAVFile(MediaFile):
    def play(self):
        print("playoing in wav")
    def stop(self):
        print("Stops Playing in wav")

mp3=MP3File('song.mp3')

def start_palyer(media):
    return media.play()
start_palyer(mp3)
print()

mp3 = MP3File('song.mp3')
mp4 = MP4File('song.mp4')
wav = WAVFile('somg.wav')
l=[mp3,mp4,wav]
for i in l:
    i.play()
    i.stop()
# mp3.play()
# mp4.play()
# wav.play()

Playing is mp3

Playing is mp3
Stops Playing in mp3
Playing in mp4
Stops Playing in mp4
playoing in wav
Stops Playing in wav
